In [ ]:
import json

# load your dataset
with open("dataset.json", "r") as f:
    data = json.load(f)

converted = []

for item in data:
    if item["type"] == "factual":
        instruction = "Provide a precise factual answer. Do not guess. If unsure, say you don't know."
    else:
        instruction = "Provide a clear and accurate explanation. Avoid speculation."

    converted.append({
        "instruction": instruction,
        "input": item["question"],
        "output": item["reference_answer"]
    })

# save new dataset
with open("finetune_data.json", "w") as f:
    json.dump(converted, f, indent=4)

print("Done!")

Done!


In [1]:
pip install transformers datasets peft accelerate bitsandbytes trl

In [2]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="final_dataset.json")["train"]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
from huggingface_hub import login


In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen3-4B-Instruct-2507"

# quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True
)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [8]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [9]:
from trl import SFTTrainer
from transformers import TrainingArguments

def formatting_func(example):
    return f"""<|system|>
You are a climate science assistant. Avoid hallucinations. If unsure, say you don't know.

<|user|>
{example['instruction']}
Question: {example['input']}

<|assistant|>
{example['output']}"""

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    bf16=True,         # Set to True
    fp16=False,        # Set to False
    optim="paged_adamw_32bit", # Optimized for 4-bit training
    logging_steps=10,
    save_strategy="no"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    formatting_func=formatting_func,
    processing_class=tokenizer,
    args=training_args
)

tokenizer.pad_token = tokenizer.eos_token

In [10]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss
10,1.573106
20,0.793469
30,0.632120
40,0.544176
50,0.465872
60,0.416306
70,0.417292
80,0.321456
90,0.273774
100,0.292680


TrainOutput(global_step=105, training_loss=0.5588122481391543, metrics={'train_runtime': 1799.12, 'train_samples_per_second': 0.917, 'train_steps_per_second': 0.058, 'total_flos': 3720513579417600.0, 'train_loss': 0.5588122481391543})

In [20]:
model.save_pretrained("final_adapter")
tokenizer.save_pretrained("final_adapter")

('final_adapter/tokenizer_config.json',
 'final_adapter/chat_template.jinja',
 'final_adapter/tokenizer.json')

In [21]:
from huggingface_hub import upload_folder

upload_folder(
    folder_path="final_adapter",
    repo_id="SumedhaBasu/qwen3-4b-climate-science",
    commit_message="Upload LoRA adapter properly"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...al_adapter/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

  ...adapter_model.safetensors: 100%|##########| 66.1MB / 66.1MB            

CommitInfo(commit_url='https://huggingface.co/SumedhaBasu/qwen3-4b-climate-science/commit/de2d38757aa5570631445dc38b914fa5c6b2eaeb', commit_message='Upload LoRA adapter properly', commit_description='', oid='de2d38757aa5570631445dc38b914fa5c6b2eaeb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/SumedhaBasu/qwen3-4b-climate-science', endpoint='https://huggingface.co', repo_type='model', repo_id='SumedhaBasu/qwen3-4b-climate-science'), pr_revision=None, pr_num=None)

In [11]:
# 1. Set your Hugging Face username and the desired repo name
hf_username = "SumedhaBasu"  # <-- CHANGE THIS to your actual HF username
repo_id = f"{hf_username}/qwen3-4b-climate-science"

# 2. Push the adapters (the weights you just trained)
# This will also create the repo if it doesn't exist
trainer.push_to_hub(repo_id)

# 3. Push the tokenizer (essential for others to use your model)
tokenizer.push_to_hub(repo_id)

print(f"Done! Your model is live at: https://huggingface.co/{repo_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...nt/results/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

  ...adapter_model.safetensors:   0%|          | 52.1kB / 66.1MB            

  ...results/training_args.bin:   1%|          |  51.0B / 5.58kB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp4t3wh67c/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Done! Your model is live at: https://huggingface.co/SumedhaBasu/qwen3-4b-climate-science


In [14]:
finetuned_model = model

In [15]:
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B-Instruct-2507",
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [35]:
def compare(question, instruction):
    prompt = f"""<|system|>
You are a climate science assistant. Answer the question.

<|user|>
{instruction}
Question: {question}

<|assistant|>"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    print("\n--- BASE MODEL ---")
    base_out = base_model.generate(**inputs, max_new_tokens=100, min_new_tokens=10)
    print(tokenizer.decode(base_out[0], skip_special_tokens=True))

    print("\n--- FINETUNED MODEL ---")
    ft_out = finetuned_model.generate(**inputs, max_new_tokens=100, min_new_tokens=10)
    print(tokenizer.decode(ft_out[0], skip_special_tokens=True))

In [17]:
compare(
    "What is the atmospheric lifetime of methane?",
    "Provide a precise factual answer. Do not guess."
)


--- BASE MODEL ---
<|system|>
You are a climate science assistant. Answer the question.

<|user|>
Provide a precise factual answer. Do not guess.
Question: What is the atmospheric lifetime of methane?

<|assistant|>  
The atmospheric lifetime of methane (CH₄) is approximately 12 to 14 years. This value refers to the average time it takes for methane to be removed from the atmosphere through chemical reactions, primarily oxidation by hydroxyl radicals (OH). According to the Intergovernmental Panel on Climate Change (IPCC) Sixth Assessment Report (AR6), the global mean lifetime of methane is about 12 years, with a range of 11 to 14 years

--- FINETUNED MODEL ---
<|system|>
You are a climate science assistant. Answer the question.

<|user|>
Provide a precise factual answer. Do not guess.
Question: What is the atmospheric lifetime of methane?

<|assistant|> 
Methane has an atmospheric lifetime of about 9 to 12 years.


In [38]:
compare(
    "What is methane concentration?",
    "Provide a precise factual answer. Do not guess."
)


--- BASE MODEL ---
<|system|>
You are a climate science assistant. Answer the question.

<|user|>
Provide a precise factual answer. Do not guess.
Question: What is methane concentration?

<|assistant|>  
Methane concentration is the amount of methane present in a given volume of air.

--- FINETUNED MODEL ---
<|system|>
You are a climate science assistant. Answer the question.

<|user|>
Provide a precise factual answer. Do not guess.
Question: What is methane concentration?

<|assistant|>  
Methane concentration is the amount of methane present in a given volume of air.


In [22]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

repo_id = "SumedhaBasu/qwen3-4b-climate-science"

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(repo_id)

# load base model
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B-Instruct-2507",
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

# load your fine-tuned adapter
model = PeftModel.from_pretrained(base_model, repo_id)

tokenizer.pad_token = tokenizer.eos_token
model.eval()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/66.1M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [34]:
prompt = """<|system|>
You are a climate science assistant. Avoid hallucinations.

<|user|>
Provide a precise factual answer. Do not guess.
Question: What is albedo concentration in 2300?

<|assistant|>"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    min_new_tokens=10,          # ✅ prevents empty output
    temperature=0.3,            # ✅ low randomness
    do_sample=True,             # ✅ needed with temperature
    top_p=0.9,
    repetition_penalty=1.1
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

<|system|>
You are a climate science assistant. Avoid hallucinations.

<|user|>
Provide a precise factual answer. Do not guess.
Question: What is albedo concentration in 2300?

<|assistant|> 
I don't know. I'm sorry, I cannot provide an accurate answer.


LLAMA

In [ ]:
from huggingface_hub import login

In [17]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

model_name = "meta-llama/Llama-3.2-1B-Instruct"

# 1. Setup Quantization (Using bfloat16 to match your Qwen success)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token # Llama 3.2 needs this

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

# 2. LoRA Config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# 3. Llama-style Prompt Formatting
def formatting_func(example):
    return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a climate science assistant. Avoid hallucinations. If unsure, say you don't know.<|eot_id|><|start_header_id|>user<|end_header_id|>

{example['instruction']}
Question: {example['input']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{example['output']}<|eot_id|>"""

# 4. Training Arguments (Same settings that worked for you)
training_args = TrainingArguments(
    output_dir="./llama32_results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    bf16=True,         # Matches your Qwen success
    fp16=False,
    optim="paged_adamw_32bit",
    logging_steps=10,
    save_strategy="no"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    formatting_func=formatting_func,
    processing_class=tokenizer,
    args=training_args
)

trainer.train()

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss
10,2.225810
20,1.215732
30,1.040196
40,0.897433
50,0.716909
60,0.599825
70,0.574809
80,0.489830
90,0.437020
100,0.448367


TrainOutput(global_step=105, training_loss=0.8424565065474737, metrics={'train_runtime': 549.1642, 'train_samples_per_second': 3.005, 'train_steps_per_second': 0.191, 'total_flos': 1007662388994048.0, 'train_loss': 0.8424565065474737})

In [18]:
finetuned_model = model

In [19]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [20]:
def compare_llama(question, instruction):
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a climate science assistant. Avoid hallucinations. If unsure, say you don't know.<|eot_id|><|start_header_id|>user<|end_header_id|>

{instruction}
Question: {question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    print("\n--- BASE MODEL ---")
    base_out = base_model.generate(
        **inputs,
        max_new_tokens=100,
        min_new_tokens=10,
        temperature=0.3,
        do_sample=True
    )
    print(tokenizer.decode(base_out[0], skip_special_tokens=True))

    print("\n--- FINETUNED MODEL ---")
    ft_out = finetuned_model.generate(
        **inputs,
        max_new_tokens=100,
        min_new_tokens=10,
        temperature=0.3,
        do_sample=True
    )
    print(tokenizer.decode(ft_out[0], skip_special_tokens=True))

In [30]:
compare_llama(
    "What is the global temperature in 2015?",
    "Provide a precise factual answer. Do not guess."
)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



--- BASE MODEL ---
system

You are a climate science assistant. Avoid hallucinations. If unsure, say you don't know.user

Provide a precise factual answer. Do not guess.
Question: What is the global temperature in 2015?assistant
It was the warmest year on record. It was also the second-warmest year globally.

--- FINETUNED MODEL ---
system

You are a climate science assistant. Avoid hallucinations. If unsure, say you don't know.user

Provide a precise factual answer. Do not guess.
Question: What is the global temperature in 2015?assistant
It was 1.1°C above pre-industrial levels.


In [22]:
model.save_pretrained("llama_adapter")
tokenizer.save_pretrained("llama_adapter")

('llama_adapter/tokenizer_config.json',
 'llama_adapter/chat_template.jinja',
 'llama_adapter/tokenizer.json')

In [24]:
from huggingface_hub import create_repo

repo_id = "SumedhaBasu/llama32-1b-climate-science"

create_repo(repo_id, exist_ok=True)

RepoUrl('https://huggingface.co/SumedhaBasu/llama32-1b-climate-science', endpoint='https://huggingface.co', repo_type='model', repo_id='SumedhaBasu/llama32-1b-climate-science')

In [25]:
from huggingface_hub import upload_folder

repo_id = "SumedhaBasu/llama32-1b-climate-science"

upload_folder(
    folder_path="llama_adapter",
    repo_id=repo_id,
    commit_message="Upload LLaMA LoRA adapter"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ma_adapter/tokenizer.json:   2%|1         |  295kB / 17.2MB            

  ...adapter_model.safetensors:   0%|          | 29.1kB / 22.6MB            

CommitInfo(commit_url='https://huggingface.co/SumedhaBasu/llama32-1b-climate-science/commit/53e41149d8bfa9ad7f442129e0ca08001edbd5c9', commit_message='Upload LLaMA LoRA adapter', commit_description='', oid='53e41149d8bfa9ad7f442129e0ca08001edbd5c9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/SumedhaBasu/llama32-1b-climate-science', endpoint='https://huggingface.co', repo_type='model', repo_id='SumedhaBasu/llama32-1b-climate-science'), pr_revision=None, pr_num=None)

In [26]:
from peft import PeftModel

repo_id = "SumedhaBasu/llama32-1b-climate-science"

tokenizer = AutoTokenizer.from_pretrained(repo_id)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

model = PeftModel.from_pretrained(base_model, repo_id)
model.eval()

tokenizer_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/22.6M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [27]:
def ask_llama(question, instruction):
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a climate science assistant. Avoid hallucinations. If unsure, say you don't know.<|eot_id|><|start_header_id|>user<|end_header_id|>

{instruction}
Question: {question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        min_new_tokens=10,
        temperature=0.3,
        do_sample=True
    )

    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [28]:
ask_llama(
    "What will be global temperature in 2100?",
    "Provide a precise factual answer. Do not guess."
)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


system

You are a climate science assistant. Avoid hallucinations. If unsure, say you don't know.user

Provide a precise factual answer. Do not guess.
Question: What will be global temperature in 2100?assistant
I don't know. It depends on many factors including emissions and solar forcing.
